In [1]:
import os
import csv
import subprocess

import pandas as pd

In [48]:
base_dir = "/home/omen/Documents/Megha/ABIDE_DATA/ABIDE_I"

output_dir = "/home/omen/Documents/Megha/ABIDE1_PREPROCESSED"

# cropped_image_folder = "/home/omen/Documents/Megha/preprocessed_ixi1/cropped"
bet_folder_loc ='/home/omen/Documents/Megha/ABIDE1_PREPROCESSED/bet'
bias_field_corrected_loc = '/home/omen/Documents/Megha/ABIDE1_PREPROCESSED/bias_field_corrected'
flirt_registered__loc= '/home/omen/Documents/Megha/ABIDE1_PREPROCESSED/linear_registered'
affine_mat__loc = '/home/omen/Documents/Megha/ABIDE1_PREPROCESSED/affine'
warp_coeff_loc ='/home/omen/Documents/Megha/ABIDE1_PREPROCESSED/warp'
non_linear_loc =  '/home/omen/Documents/Megha/ABIDE1_PREPROCESSED/non_linear_registered'
# complete_preprocessed_loc = '/home/omen/Documents/Megha/ABIDE1_PREPROCESSED/completely_preprocessed'

stages = {
    "bet":        bet_folder_loc,
    "bias_field": bias_field_corrected_loc,
    "flirt":      flirt_registered__loc,
    "affine":     affine_mat__loc,
    "warp":       warp_coeff_loc,
    "non_linear": non_linear_loc
    # "final_bet":  complete_preprocessed_loc,
    # "cropped_image" : cropped_image_folder
}


config_file = "T1_2_MNI152_2mm"
FSL_STANDARD_BRAIN = '/home/omen/fsl5.0.10/data/standard/MNI152_T1_2mm.nii.gz'
FSL_STANDARD_BRAIN_MASKED = '/home/omen/fsl5.0.10/data/standard/MNI152_T1_2mm_brain.nii.gz'
f_value= 0.4


loading the csv for the data

In [49]:
file = pd.read_csv('/home/omen/Documents/Megha/ABIDE_I_relative_paths_with_age_sex.csv')
print(file.columns)
for rel_path in file['relative_path']:
    site = rel_path.split("/")[0]
    # site = os.path.basename(rel_path)
    # site = os.path.basename(rel_path).replace(".nii.gz", "")
    # t1_files = file[file['relative_path'].str.contains("T1", case=False)]
    print(site)

for rel_path in file['relative_path']:
    site = rel_path.split("/")[0]
    # print(site)
    out_fname = f"{site}.nii.gz" 
    print(out_fname)

Index(['subject', 'relative_path', 'age_at_scan', 'sex'], dtype='object')
CMU_50665
Leuven_50707
OHSU_50160
Yale_50577
SDSU_50198
NYU_51081
Trinity_50263
KKI_50790
Leuven_50723
Trinity_51140
OHSU_50158
Yale_50567
KKI_50773
SDSU_50197
SBL_51566
Olin_50112
SDSU_50211
Pitt_50030
NYU_51122
NYU_51067
NYU_51096
MaxMun_51373
Trinity_50260
Caltech_51490
UM_50427
UM_50428
KKI_50781
MaxMun_51360
Yale_50565
UM_50334
NYU_51044
Pitt_50044
CMU_50656
NYU_51084
NYU_51153
Caltech_51480
NYU_51069
Yale_50556
NYU_51126
USM_50441
NYU_51151
Pitt_50050
UM_50370
KKI_50814
SDSU_50200
SBL_51562
UM_50391
KKI_50778
UM_50371
Stanford_51182
KKI_50819
USM_50447
UM_50381
Pitt_50043
USM_50462
OHSU_50165
UM_50367
NYU_51051
Stanford_51193
Olin_50107
Leuven_50726
OHSU_50163
Pitt_50036
CMU_50658
UM_50374
Pitt_50060
NYU_51075
Leuven_50738
UM_50358
NYU_51149
Leuven_50727
Leuven_50731
NYU_51114
UM_50383
UCLA_51304
Leuven_50732
UM_50345
SDSU_50193
NYU_51045
NYU_51056
Caltech_51483
UM_50416
Trinity_51137
SDSU_50194
UCLA_51261


In [ ]:

# Loop through the filtered T1 file paths
for rel_path in file['relative_path']:
    site = rel_path.split("/")[0]
    out_fname = f"{site}.nii.gz"  # ABIDE_KKI_29473.nii.gz
    paths = {
            "bet":        os.path.join(stages["bet"], out_fname),
            "bias_field" : os.path.join(stages["bias_field"],site),
            "flirt":      os.path.join(stages["flirt"],out_fname),
            "affine":     os.path.join(stages["affine"], f"{site}.mat"),
            "warp":       os.path.join(stages["warp"],  out_fname),
            "non_linear": os.path.join(stages["non_linear"],out_fname)
            # "final_bet":  os.path.join(stages["final_bet"], out_fname),
            # "cropped_image": os.path.join(stages["cropped_image"],out_fname)
        }
    for stage, p in paths.items():
        if stage== "bias_field":
            os.makedirs(os.path.dirname(p), exist_ok=True)
        else:
            os.makedirs(os.path.dirname(p), exist_ok=True)
    input_image = os.path.join(base_dir, rel_path)
    print(input_image)
    
    try:
        # print("cropping image")
        # subprocess.run(["robustfov", "-i", input_image, "-r", paths["cropped_image"]], check=True)

        print(f"running bet for {input_image}")
        subprocess.run([
            "bet",
            input_image,
            paths["bet"],
            "-f", str(f_value),
            "-g", "0",
            "-R"  # robust brain center estimate
             
                ], check=True)
        if not os.path.exists(paths["bet"]):
            print(f" BET did not produce output for: {input_image}")
            continue
        print("running_bias_field_correction")
        bias_base = paths["bias_field"]  
        print("-> running FAST (bias correction), output base:", bias_base)
        subprocess.run([
            "fast",
            "-B",           # output bias corrected image
            "-t", "1",      # image type = T1
            "-o", bias_base,  # output base name
            paths["bet"]
        ],check=True)
        bias_restore = bias_base + "_restore.nii.gz"
        if not os.path.exists(bias_restore):
            print("Expected FAST output not found:", bias_restore)
            print("Listing bias_field dir for debugging:", os.listdir(os.path.dirname(bias_base)))
            continue
        print("running_linear_registeration")
        
        subprocess.run([
                    "flirt",
                    "-ref", FSL_STANDARD_BRAIN_MASKED,
                    "-in", bias_restore,
                    "-omat", paths["affine"],    #change the affine - wrt the file extension
                    "-out", paths["flirt"]
                ], check=True)
        print("running non linear registeration")
        subprocess.run([
            "fnirt",
            "--in=" + bias_restore,
            "--aff=" + paths["affine"],
            "--cout=" + paths["warp"],
            "--config=" + config_file
        ], check=True)
        print("applying the transformnation")
        subprocess.run([
            "applywarp",
            "--ref=" + FSL_STANDARD_BRAIN_MASKED,
            "--in=" + bias_restore,
            "--warp=" + paths["warp"],
            "--out=" + paths["non_linear"]
        ], check=True)
        # print("lastly applying bet again on the final non linear registered image")
        # subprocess.run([
        #     "bet",
        #     paths["non_linear"],
        #     paths["final_bet"],

        # ], check=True)
        

    except subprocess.CalledProcessError as e:
                print(f"Failed processing {input_image}: {e}")
    


/home/omen/Documents/Megha/ABIDE_DATA/ABIDE_I/CMU_50665/CMU_50665/anat/NIfTI/mprage.nii.gz
running bet for /home/omen/Documents/Megha/ABIDE_DATA/ABIDE_I/CMU_50665/CMU_50665/anat/NIfTI/mprage.nii.gz
running_bias_field_correction
-> running FAST (bias correction), output base: /home/omen/Documents/Megha/ABIDE1_PREPROCESSED/bias_field_corrected/CMU_50665
running_linear_registeration
running non linear registeration
applying the transformnation
/home/omen/Documents/Megha/ABIDE_DATA/ABIDE_I/Leuven_50707/Leuven_50707/anat/NIfTI/mprage.nii.gz
running bet for /home/omen/Documents/Megha/ABIDE_DATA/ABIDE_I/Leuven_50707/Leuven_50707/anat/NIfTI/mprage.nii.gz
running_bias_field_correction
-> running FAST (bias correction), output base: /home/omen/Documents/Megha/ABIDE1_PREPROCESSED/bias_field_corrected/Leuven_50707
running_linear_registeration
running non linear registeration
applying the transformnation
/home/omen/Documents/Megha/ABIDE_DATA/ABIDE_I/OHSU_50160/OHSU_50160/anat/NIfTI/mprage.nii.gz
ru

Brain extraction

In [43]:
input = '/home/omen/Documents/Megha/ABIDE_DATA/ABIDE_I/OHSU_50158/OHSU_50158/anat/NIfTI/mprage.nii.gz'
output = 'brain_extracted.nii.gz'
f_value= 0.4

In [44]:
subprocess.run([
            "bet",
            input,
            output,
            "-f", str(f_value),
            "-g", "0",
            "-R",  # robust brain center estimate
            "-m"   # save mask for inspection   
                ], check=True)
        


CompletedProcess(args=['bet', '/home/omen/Documents/Megha/ABIDE_DATA/ABIDE_I/OHSU_50158/OHSU_50158/anat/NIfTI/mprage.nii.gz', 'brain_extracted.nii.gz', '-f', '0.4', '-g', '0', '-R', '-m'], returncode=0)

In [15]:
import subprocess

# Define inputs and outputs
input_file = "/home/omen/Documents/Megha/FSL_PREPROCESSING/brain_extracted.nii.gz"
output_basename = "brain_bias_field"

# Construct the FSL FAST command
cmd = [
    "fast",
    "-B",           # output bias corrected image
    "-t", "1",      # image type = T1
    "-o", output_basename,  # output base name
    input_file
]

# Run the command
subprocess.run(cmd, check=True)

print("Bias field correction completed. Outputs saved with prefix:", output_basename)


Bias field correction completed. Outputs saved with prefix: brain_bias_field


In [16]:
config_file = "T1_2_MNI152_2mm"
FSL_STANDARD_BRAIN = '/home/omen/fsl5.0.10/data/standard/MNI152_T1_2mm.nii.gz'
FSL_STANDARD_BRAIN_MASKED = '/home/omen/fsl5.0.10/data/standard/MNI152_T1_2mm_brain.nii.gz'

In [17]:
input='/home/omen/Documents/Megha/FSL_PREPROCESSING/brain_bias_field_restore.nii.gz'
affine = 'affine.mat'
out = 'linearly_registered.nii.gz'

In [18]:
subprocess.run([
                    "flirt",
                    "-ref", FSL_STANDARD_BRAIN_MASKED,
                    "-in", input,
                    "-omat",affine ,    #change the affine - wrt the file extension
                    "-out", out
                ], check=True)
        

CompletedProcess(args=['flirt', '-ref', '/home/omen/fsl5.0.10/data/standard/MNI152_T1_2mm_brain.nii.gz', '-in', '/home/omen/Documents/Megha/FSL_PREPROCESSING/brain_bias_field_restore.nii.gz', '-omat', 'affine.mat', '-out', 'linearly_registered.nii.gz'], returncode=0)

In [19]:
input = '/home/omen/Documents/Megha/FSL_PREPROCESSING/brain_bias_field_restore.nii.gz'
affine = '/home/omen/Documents/Megha/FSL_PREPROCESSING/affine.mat'
warp = 'warp.nii.gz'
output = 'non_linear.nii.gz'

In [20]:
subprocess.run([
            "fnirt",
            "--in=" + input,
            "--aff=" + affine,
            "--cout=" + warp,
            "--config=" + config_file
        ], check=True)
print("applying the transformnation")
subprocess.run([
    "applywarp",
    "--ref=" + FSL_STANDARD_BRAIN_MASKED,
    "--in=" + input,
    "--warp=" + warp,
    "--out=" +output
], check=True)

applying the transformnation


CompletedProcess(args=['applywarp', '--ref=/home/omen/fsl5.0.10/data/standard/MNI152_T1_2mm_brain.nii.gz', '--in=/home/omen/Documents/Megha/FSL_PREPROCESSING/brain_bias_field_restore.nii.gz', '--warp=warp.nii.gz', '--out=non_linear.nii.gz'], returncode=0)